# Subpopulation Analysis — Population-Fraction Sweep

Runs the population-fraction sweep across all datasets in `DATASETS_TO_RUN`,
measuring how well full-population decoding is preserved as a function of the
fraction of neurons kept under various neuron-selection strategies.

For interactive subpopulation inspection, see `08_encoding_decoding_explorer.ipynb`.

## Config

In [1]:
import logging
import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
from tueplots import bundles

import sys; _d = os.path.abspath(os.getcwd()); sys.path.insert(0, _d if os.path.isdir(os.path.join(_d, 'src')) else os.path.dirname(_d))
from src.subpop_utils import (
    compute_dynamic_metrics, select_top_k_by_metric,
    compute_decoding_manifold, knn_decoding_accuracy,
    procrustes_r2, variance_reproduced, rdm_correlation, linear_cka,
)
from src.metrics import compute_osi_and_pref_stim

logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
# Suppress RuntimeWarnings from PCA (randomized SVD on rank-deficient tiny subsets)
# and linear_cka (near-zero kernel matrices at k=1–3 neurons). Non-critical: results
# at those fractions are NaN-handled by np.nanmean in run_sweep.
warnings.filterwarnings('ignore', category=RuntimeWarning)
plt.rcParams.update(bundles.neurips2021(usetex=False))

In [2]:
# Complete registry of known datasets and their low_sf flag.
# Do not comment out entries here — edit DATASETS_TO_RUN below to select a subset.
_LAYER_PARAMS = {
    # ResNet-50 (shifted)
    "res50_shifted_act_i3_n1000_SCL0_5_TL37_layer1_maxFr_maxNr_seed1": dict(low_sf=False),
    "res50_shifted_act_i3_n1000_SCL0_5_TL37_layer4_maxFr_maxNr_seed1": dict(low_sf=False),
    # FNN07
    "fnn07_act_i3_n2000_SCL0_7_TL37_inputs0_maxFr_maxNr_seed1":      dict(low_sf=True),
    "fnn07_act_i3_n2000_SCL0_7_TL37_blocks0_maxFr_maxNr_seed1":      dict(low_sf=False),
    "fnn07_act_i3_n2000_SCL0_7_TL37_inputs1_maxFr_maxNr_seed1":      dict(low_sf=False),
    "fnn07_act_i3_n2000_SCL0_7_TL37_blocks1_maxFr_maxNr_seed1":      dict(low_sf=False),
    "fnn07_act_i3_n2000_SCL0_7_TL37_inputs2_maxFr_maxNr_seed1":      dict(low_sf=True),
    "fnn07_act_i3_n2000_SCL0_7_TL37_blocks2_maxFr_maxNr_seed1":      dict(low_sf=True),
    "fnn07_act_i3_n2000_SCL0_7_TL37_hidden_maxFr_maxNr_seed1":       dict(low_sf=False),
    "fnn07_act_i3_n2000_SCL0_7_TL37_recurrentout_maxFr_maxNr_seed1": dict(low_sf=False),
    "fnn07_act_i3_n2000_SCL0_7_TL37_position_maxFr_maxNr_seed1":     dict(low_sf=False),
    "fnn07_seed2": dict(low_sf=False),
    # Biological
    "Retina": dict(low_sf=True),
    "V1":     dict(low_sf=True),
    # Flyvision
    "flyvis_Retina_i3_n400_model000":  dict(low_sf=True),
    "flyvis_Lamina_i3_n350_model000":  dict(low_sf=True),
    "flyvis_Medulla_i3_n550_model000": dict(low_sf=True),
    "flyvis_T_Tm_i3_n1700_model000":   dict(low_sf=True),
}

basedir_data = '../data/sampled'
basedir_fig  = '../fig'

# Edit this list to select which datasets to run.
# To run all: DATASETS_TO_RUN = list(_LAYER_PARAMS.keys())
DATASETS_TO_RUN = [
    "res50_shifted_act_i3_n1000_SCL0_5_TL37_layer1_maxFr_maxNr_seed1",
    "res50_shifted_act_i3_n1000_SCL0_5_TL37_layer4_maxFr_maxNr_seed1",
    "fnn07_act_i3_n2000_SCL0_7_TL37_blocks0_maxFr_maxNr_seed1",
    "fnn07_act_i3_n2000_SCL0_7_TL37_blocks1_maxFr_maxNr_seed1",
    "fnn07_act_i3_n2000_SCL0_7_TL37_blocks2_maxFr_maxNr_seed1",
    "fnn07_act_i3_n2000_SCL0_7_TL37_hidden_maxFr_maxNr_seed1",
    "fnn07_act_i3_n2000_SCL0_7_TL37_position_maxFr_maxNr_seed1",
    "fnn07_seed2",
    "Retina",
    "V1",
    "flyvis_Retina_i3_n400_model000",
    "flyvis_Lamina_i3_n350_model000",
    "flyvis_Medulla_i3_n550_model000",
    "flyvis_T_Tm_i3_n1700_model000",
]

In [3]:
def load_dataset_for_sweep(PREFIX):
    """Load one dataset for the fraction sweep.

    Returns: tensor4d, metrics, NSTIMS, NDIRS
    """
    low_sf = _LAYER_PARAMS[PREFIX]['low_sf']

    tensor4d = np.load(f'{basedir_data}/tensor4d_{PREFIX}.npy')
    tensor4d = tensor4d - np.min(tensor4d)
    if low_sf:
        tensor4d = tensor4d[:, :6]
    else:
        tensor4d = tensor4d[:, np.array([0, 6, 7, 8, 9, 10])]
    N, NSTIMS, NDIRS, TRIAL_LEN = tensor4d.shape

    metrics = compute_dynamic_metrics(tensor4d, target_stim_idx=None)
    osi_vals, _ = compute_osi_and_pref_stim(
        tensor4d.reshape(N, NSTIMS, NDIRS * TRIAL_LEN), n_stim=NSTIMS, n_dir=NDIRS)
    metrics['osi'] = osi_vals

    print(f'  {PREFIX}: N={N}  NSTIMS={NSTIMS}  NDIRS={NDIRS}')
    return tensor4d, metrics, NSTIMS, NDIRS

## Section 6: Population-Fraction Sweep (Multi-Dataset) <a id='section-6'></a>

Runs the population-fraction sweep across **all datasets** in `DATASETS_TO_RUN`.
For each dataset, `load_dataset_for_sweep()` handles tensor loading, CP decomposition,
PCA, and outlier removal; `run_sweep()` performs the sweep; `save_sweep_plot()` saves
a per-dataset PDF to `fig/subpop/{PREFIX}_fraction_sweep.pdf`.

**Metrics (5 panels):**
- **k-NN accuracy** — can stimuli still be classified from the sub-population manifold?
- **Procrustes R²** — geometric similarity to the full-population manifold (rotation/scale-invariant); NaN at tiny k fixed by zero-padding to 3D
- **Variance reproduced** — ratio of manifold spread (subpop / full); >1 is possible for noisy small subsets
- **RDM correlation (ρ)** — Spearman rank correlation of pairwise stimulus-distance matrices in raw neural space; most interpretable
- **Linear CKA** — centered kernel alignment in stimulus space; invariant to rotation AND isotropic scaling

**Strategies (13 total — all paired hi/lo except random):**
- `random` — random subset, mean ± 1 std over `N_RANDOM_SEEDS` seeds (null hypothesis)
- `speed / [lo]` — highest / lowest early-transient magnitude
- `curvature / [lo]` — highest / lowest nonlinearity of early response
- `stability / [lo]` — highest / lowest late-transient magnitude
- `classif. / [lo]` — highest / lowest stimulus discriminability
- `osi / [lo]` — highest / lowest orientation selectivity index
- `pc_contrib / [lo]` — highest / lowest loading on first global PC

> **Sanity checks:** All metrics should equal ~1.0 at f=1.0 for every dataset.
> Results are stored in `all_results[PREFIX]` and `all_fractions[PREFIX]`.
> Per-dataset PDFs are saved immediately after each sweep to prevent memory accumulation.

In [4]:
# ── Section 6 config ────────────────────────────────────────────────────────
FRACTIONS = np.array([0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40,
                      0.50, 0.60, 0.70, 0.80, 0.90, 1.00])
N_RANDOM_SEEDS = 10
rng = np.random.default_rng(0)

STRATEGIES = {
    # baseline
    'random':          {'type': 'random'},
    # temporal dynamics
    'speed':           {'type': 'metric', 'metric': 'speed',           'high': True},
    'speed [lo]':      {'type': 'metric', 'metric': 'speed',           'high': False},
    'curvature':       {'type': 'metric', 'metric': 'curvature',       'high': True},
    'curvature [lo]':  {'type': 'metric', 'metric': 'curvature',       'high': False},
    'stability':       {'type': 'metric', 'metric': 'stability',       'high': True},
    'stability [lo]':  {'type': 'metric', 'metric': 'stability',       'high': False},
    # stimulus selectivity
    'classif.':        {'type': 'metric', 'metric': 'classifiability', 'high': True},
    'classif. [lo]':   {'type': 'metric', 'metric': 'classifiability', 'high': False},
    'osi':             {'type': 'metric', 'metric': 'osi',             'high': True},
    'osi [lo]':        {'type': 'metric', 'metric': 'osi',             'high': False},
    # representational geometry
    'pc_contrib':      {'type': 'metric', 'metric': 'pc_contrib',      'high': True},
    'pc_contrib [lo]': {'type': 'metric', 'metric': 'pc_contrib',      'high': False},
}


def _pad_to_3d(coords, ref_shape):
    """Zero-pad coords to match ref_shape[1] columns (fixes Procrustes NaN at small k)."""
    if coords.shape[1] < ref_shape[1]:
        pad = np.zeros((coords.shape[0], ref_shape[1] - coords.shape[1]))
        return np.hstack([coords, pad])
    return coords


def run_sweep(tensor4d, metrics, NSTIMS, NDIRS):
    """Run the population-fraction sweep for one dataset.
    Returns: (results dict, FRACTIONS_SWEEP array)
    """
    N = tensor4d.shape[0]
    stim_labels_full = np.repeat(np.arange(NSTIMS), NDIRS)

    _single_frac    = np.array([1, 2, 3, 5, 8, 13, 20]) / N
    FRACTIONS_SWEEP = np.unique(np.concatenate([_single_frac, FRACTIONS]))

    coords_full, _ = compute_decoding_manifold(tensor4d, n_components=3)

    results = {name: {'acc': [], 'r2': [], 'var': [], 'rdm': [], 'cka': []}
               for name in STRATEGIES}

    for name, cfg in STRATEGIES.items():
        for f in FRACTIONS_SWEEP:
            k      = max(1, int(round(f * N)))
            n_comp = min(3, k)

            if cfg['type'] == 'random':
                accs, r2s, vars_, rdms, ckas = [], [], [], [], []
                for seed in range(N_RANDOM_SEEDS):
                    idx    = rng.choice(N, k, replace=False)
                    t_sub  = tensor4d[idx]
                    cs, _  = compute_decoding_manifold(t_sub, n_components=n_comp)
                    cs_pad = _pad_to_3d(cs, coords_full.shape)
                    accs.append(knn_decoding_accuracy(cs, stim_labels_full))
                    r2s.append(procrustes_r2(coords_full, cs_pad))
                    vars_.append(variance_reproduced(coords_full, cs))
                    rdms.append(rdm_correlation(tensor4d, t_sub))
                    ckas.append(linear_cka(tensor4d, t_sub))
                results[name]['acc'].append((np.nanmean(accs), np.nanstd(accs)))
                results[name]['r2'].append((np.nanmean(r2s),   np.nanstd(r2s)))
                results[name]['var'].append((np.nanmean(vars_), np.nanstd(vars_)))
                results[name]['rdm'].append((np.nanmean(rdms), np.nanstd(rdms)))
                results[name]['cka'].append((np.nanmean(ckas), np.nanstd(ckas)))
            else:
                idx = select_top_k_by_metric(metrics, cfg['metric'], k=k, high=cfg['high'])
                if len(idx) < 1:
                    for key in ('acc', 'r2', 'var', 'rdm', 'cka'):
                        results[name][key].append((np.nan, 0.0))
                    continue
                t_sub  = tensor4d[idx]
                cs, _  = compute_decoding_manifold(t_sub, n_components=min(3, len(idx)))
                cs_pad = _pad_to_3d(cs, coords_full.shape)
                results[name]['acc'].append((knn_decoding_accuracy(cs, stim_labels_full), 0.0))
                results[name]['r2'].append((procrustes_r2(coords_full, cs_pad), 0.0))
                results[name]['var'].append((variance_reproduced(coords_full, cs), 0.0))
                results[name]['rdm'].append((rdm_correlation(tensor4d, t_sub), 0.0))
                results[name]['cka'].append((linear_cka(tensor4d, t_sub), 0.0))

    return results, FRACTIONS_SWEEP


print('Sweep config and run_sweep() ready.')
print(f'Strategies: {list(STRATEGIES)}')

Sweep config and run_sweep() ready.
Strategies: ['random', 'speed', 'speed [lo]', 'curvature', 'curvature [lo]', 'stability', 'stability [lo]', 'classif.', 'classif. [lo]', 'osi', 'osi [lo]', 'pc_contrib', 'pc_contrib [lo]']


In [5]:
_COLORS = {
    'random':          '#607d8b',
    'speed':           '#e64a18',
    'speed [lo]':      '#ffab91',
    'curvature':       '#fbc02c',
    'curvature [lo]':  '#fff176',
    'stability':       '#00b050',
    'stability [lo]':  '#a5d6a7',
    'classif.':        '#0488d1',
    'classif. [lo]':   '#81d4fa',
    'osi':             '#ff5722',
    'osi [lo]':        '#ffccbc',
    'pc_contrib':      '#9c27b0',
    'pc_contrib [lo]': '#e1bee7',
}

_PANELS = [
    ('acc', 'k-NN accuracy',        'Decoding accuracy'),
    ('r2',  'Procrustes R²',        'Geometric fidelity (Procrustes)'),
    ('var', 'Variance reproduced',  'Variance reproduced'),
    ('rdm', 'RDM correlation (ρ)',  'Stimulus-similarity structure (RSA)'),
    ('cka', 'Linear CKA',           'Representational alignment (CKA)'),
]


def save_sweep_plot(results, FRACTIONS_SWEEP, PREFIX):
    """Plot and save the fraction-sweep figure for one dataset."""
    fig, axes = plt.subplots(1, len(_PANELS), figsize=(14, 2.8), sharey=False)

    for name, res in results.items():
        color = _COLORS.get(name, 'gray')
        ls    = '--' if name == 'random' else '-'
        lw    = 1.5 if not name.endswith('[lo]') else 1.0
        alpha = 1.0 if not name.endswith('[lo]') else 0.6

        for ax, (key, ylabel, title) in zip(axes, _PANELS):
            vals = np.array([v[0] for v in res[key]])
            std  = np.array([v[1] for v in res[key]])
            ax.plot(FRACTIONS_SWEEP, vals, color=color, lw=lw, ls=ls,
                    alpha=alpha, label=name)
            if name == 'random':
                ax.fill_between(FRACTIONS_SWEEP, vals - std, vals + std,
                                color=color, alpha=0.15)

    for ax, (key, ylabel, title) in zip(axes, _PANELS):
        ax.set_xlabel('Fraction of neurons selected', fontsize=7)
        ax.set_ylabel(ylabel, fontsize=7)
        ax.set_title(title, fontsize=8)
        ax.set_xlim(1.0, FRACTIONS_SWEEP[0])   # inverted: full pop on left
        ax.tick_params(labelsize=6)
        if key in ('r2', 'cka', 'rdm'):
            ax.set_ylim(bottom=0)

    axes[2].axhline(1.0, color='black', lw=0.6, ls=':', alpha=0.5)
    axes[3].axhline(1.0, color='black', lw=0.6, ls=':', alpha=0.5)
    axes[4].axhline(1.0, color='black', lw=0.6, ls=':', alpha=0.5)
    axes[0].legend(fontsize=5.5, loc='lower left', ncol=2)

    fig.suptitle(f'Population-fraction sweep — {PREFIX}', fontsize=9)
    fig.tight_layout()
    os.makedirs(f'{basedir_fig}/subpop', exist_ok=True)
    fig.savefig(f'{basedir_fig}/subpop/{PREFIX}_fraction_sweep.pdf',
                format='pdf', bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved → {basedir_fig}/subpop/{PREFIX}_fraction_sweep.pdf')


print('save_sweep_plot() ready.')

save_sweep_plot() ready.


In [6]:
import time

all_results   = {}
all_fractions = {}

for PREFIX in DATASETS_TO_RUN:
    print(f'\n=== {PREFIX} ===')
    t0 = time.time()
    tensor4d, metrics, NSTIMS, NDIRS = load_dataset_for_sweep(PREFIX)
    results, FRACTIONS_SWEEP = run_sweep(tensor4d, metrics, NSTIMS, NDIRS)
    save_sweep_plot(results, FRACTIONS_SWEEP, PREFIX)
    all_results[PREFIX]   = results
    all_fractions[PREFIX] = FRACTIONS_SWEEP
    print(f'  Done in {time.time()-t0:.1f} s')

print(f'\nAll datasets complete. Results in all_results ({len(all_results)} datasets).')


=== res50_shifted_act_i3_n1000_SCL0_5_TL37_layer1_maxFr_maxNr_seed1 ===
  res50_shifted_act_i3_n1000_SCL0_5_TL37_layer1_maxFr_maxNr_seed1: N=1000  NSTIMS=6  NDIRS=8


/var/folders/l6/mwg72rgn22bd0l2pn599jkch0000gn/T/ipykernel_89964/265342855.py:60: UserWarning: The figure layout has changed to tight
  fig.tight_layout()


  Saved → ../fig/subpop/res50_shifted_act_i3_n1000_SCL0_5_TL37_layer1_maxFr_maxNr_seed1_fraction_sweep.pdf
  Done in 21.0 s

=== res50_shifted_act_i3_n1000_SCL0_5_TL37_layer4_maxFr_maxNr_seed1 ===
  res50_shifted_act_i3_n1000_SCL0_5_TL37_layer4_maxFr_maxNr_seed1: N=1000  NSTIMS=6  NDIRS=8


/var/folders/l6/mwg72rgn22bd0l2pn599jkch0000gn/T/ipykernel_89964/265342855.py:60: UserWarning: The figure layout has changed to tight
  fig.tight_layout()


  Saved → ../fig/subpop/res50_shifted_act_i3_n1000_SCL0_5_TL37_layer4_maxFr_maxNr_seed1_fraction_sweep.pdf
  Done in 19.4 s

=== fnn07_act_i3_n2000_SCL0_7_TL37_blocks0_maxFr_maxNr_seed1 ===
  fnn07_act_i3_n2000_SCL0_7_TL37_blocks0_maxFr_maxNr_seed1: N=2000  NSTIMS=6  NDIRS=8


...

  Saved → ../fig/subpop/flyvis_Lamina_i3_n350_model000_fraction_sweep.pdf
  Done in 18.5 s

=== flyvis_Medulla_i3_n550_model000 ===
  flyvis_Medulla_i3_n550_model000: N=550  NSTIMS=6  NDIRS=8


/var/folders/l6/mwg72rgn22bd0l2pn599jkch0000gn/T/ipykernel_89964/265342855.py:60: UserWarning: The figure layout has changed to tight
  fig.tight_layout()


  Saved → ../fig/subpop/flyvis_Medulla_i3_n550_model000_fraction_sweep.pdf
  Done in 19.8 s

=== flyvis_T_Tm_i3_n1700_model000 ===
  flyvis_T_Tm_i3_n1700_model000: N=1700  NSTIMS=6  NDIRS=8


/var/folders/l6/mwg72rgn22bd0l2pn599jkch0000gn/T/ipykernel_89964/265342855.py:60: UserWarning: The figure layout has changed to tight
  fig.tight_layout()


  Saved → ../fig/subpop/flyvis_T_Tm_i3_n1700_model000_fraction_sweep.pdf
  Done in 26.0 s

All datasets complete. Results in all_results (14 datasets).
